In [ ]:

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="mllab/alfa_ct",
    local_dir="./alfa_ct_data",          # папка для сохранения
    repo_type="dataset",
    max_workers=8,                       # количество параллельных потоков
    allow_patterns=["*.parquet"]         # скачиваем только parquet-файлы
)

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import numpy as np
import polars as pl

# 1. Загружаем данные (все .parquet файлы)
df = pl.read_parquet("alfa_ct_data/*.parquet")

# 2. Преобразуем в pandas для удобства (если данных не слишком много)
df = df.to_pandas()

# 3. Функция агрегации для одного клиента
def aggregate_transactions(row):
    trans = row['transactions']
    if not trans or len(trans) == 0:
        # Если транзакций нет — заполняем нулями
        return {
            'app_id': row['app_id'],
            'avg_amount': 0,
            'max_amount': 0,
            'std_amount': 0,
            'sum_amount': 0,
            'n_transactions': 0,
            'n_unique_mcc': 0,
            'night_ratio': 0,
            'avg_days_before': 0,
            'trans_last_30d': 0,
            'ecommerce_ratio': 0,
            'flag': row['flag']
        }
    
    # Превращаем список транзакций в массив
    arr = np.array(trans)
    
    # Извлекаем нужные колонки по индексам
    amounts = arr[:, 0]                     # сумма
    mcc = arr[:, 10] if arr.shape[1] > 10 else np.zeros(len(trans))  # категория MCC
    hours = arr[:, 15] if arr.shape[1] > 15 else np.zeros(len(trans)) # час
    days_before = arr[:, 16] if arr.shape[1] > 16 else np.zeros(len(trans)) # дней до выдачи
    ecommerce = arr[:, 7] if arr.shape[1] > 7 else np.zeros(len(trans)) # e-commerce флаг
    
    return {
        'app_id': row['app_id'],
        'avg_amount': np.mean(amounts),
        'max_amount': np.max(amounts),
        'std_amount': np.std(amounts) if len(amounts) > 1 else 0,
        'sum_amount': np.sum(amounts),
        'n_transactions': len(trans),
        'n_unique_mcc': len(set(mcc)),
        'night_ratio': np.mean((hours < 6) | (hours > 22)) if len(hours) > 0 else 0,
        'avg_days_before': np.mean(days_before) if len(days_before) > 0 else 0,
        'trans_last_30d': np.sum(days_before <= 30) if len(days_before) > 0 else 0,
        'ecommerce_ratio': np.mean(ecommerce) if len(ecommerce) > 0 else 0,
        'flag': row['flag']
    }

# 4. Применяем ко всем клиентам
aggregated = []
for idx, row in df.iterrows():
    aggregated.append(aggregate_transactions(row))

# 5. Превращаем в DataFrame
flat_df = pd.DataFrame(aggregated)

# 6. Сохраняем результат
flat_df.to_csv("flattened_alfa_data.csv", index=False)
print(f"✅ Сохранено {len(flat_df)} клиентов в flattened_alfa_data.csv") 

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,        # количество деревьев (итераций бустинга)
    learning_rate=0.05,    # шаг обучения (скорость адаптации)
    depth=6,               # максимальная глубина каждого дерева
    verbose=100,           # выводить прогресс каждые 100 итераций
    random_seed=42         # фиксируем случайность для воспроизводимости
)
model.fit(X_train, y_train, eval_set=(X_val, y_val))

In [ ]:
# Для всей выборки (или для валидационной)
proba = model.predict_proba(X)[:, 1]   # вероятность класса 1 (дефолт)

In [ ]:
import numpy as np

# Возводим матрицу в 12-ю степень
P_12 = np.linalg.matrix_power(P, 12)

# PD для клиента в состоянии 0 (платит)
pd_12m = P_12[0, 2]

# ECL
LGD = 0.45  # типичное значение для необеспеченных кредитов
EAD = 10000 # средний кредитный лимит (можно взять из данных)

ecl = pd_12m * LGD * EAD
print(f"PD за год: {pd_12m:.4f}")
print(f"ECL: {ecl:.2f}")

In [ ]:
from sklearn.metrics import roc_auc_score

# Предсказания на валидационной выборке
y_pred_proba = model.predict_proba(X_val)[:, 1]

# ROC-AUC
auc = roc_auc_score(y_val, y_pred_proba)
print(f"ROC-AUC на валидации: {auc:.4f}")

In [ ]:
# Вероятности дефолта для всей выборки
proba = model.predict_proba(X)[:, 1]

МАТРИЦА ПЕРЕХОДОВ 

In [ ]:
import numpy as np

# Средняя вероятность дефолта по всем клиентам
p_avg = np.mean(proba)

# Матрица переходов за 1 месяц (состояния: 0 - платит, 1 - дефолт)
P = np.array([
    [1 - p_avg, p_avg],
    [0, 1]
])

print("Матрица переходов за месяц:")
print(P)

PD - матрица переходов в 12-й степени 

In [ ]:
# Возводим матрицу в 12-ю степень
P_12 = np.linalg.matrix_power(P, 12)

# PD за год
pd_12m = P_12[0, 1]  # для 2x2
# или P_12[0, 2]  # для 3x3

print(f"Вероятность дефолта за год (PD): {pd_12m:.4f}")

Что делаем:
ECL = PD × LGD × EAD

In [ ]:
LGD = 0.45      # Loss Given Default (типичное значение 45%)
EAD = 10000     # Exposure at Default (средний кредитный лимит)

ecl = pd_12m * LGD * EAD
print(f"Expected Credit Loss (ECL): {ecl:.2f}")